# Proyecto Sprint 3 - Análisis del desempeño financiero de Adventure Works con SQL

**Introducción**

Eres analista en AdventureWorks. El director financiero quiere saber en qué mercados se generan más ingresos y rentabilidad para decidir dónde invertir el próximo dólar de marketing.

Con datos de órdenes, productos, territorios y campañas, tu tarea es preparar un análisis que muestre prioridades de mercado, optimización de presupuesto y rentabilidad.

**Objetivos del proyecto**

Al finalizar el proyecto podrás:

1. Navegar un esquema relacional y escribir JOINs para combinar tablas.
2. Extraer, filtrar y limpiar datos con SQL (manejo de NULLs, casting de tipos, estandarización de categorías).
3. Calcular indicadores financieros clave: ingresos, costos, beneficio bruto, margen y ROI.
4. Validar y controlar calidad (QA) con comprobaciones de totales y márgenes.
5. Redactar un informe ejecutivo con visualizaciones y el método Contexto → Hallazgo → Implicación (C→F→I).

**Dataset del proyecto**

**Tablas Disponibles**

Usaremos un subconjunto del dataset de AdventureWorks. Estas son las tablas que están disponibles para ti:

**ventas_2017:** transacciones de líneas de pedido (2017). Grano: una línea por producto y pedido.
**productos:** catálogo con atributos, costo y precio unitario por ClaveProducto.
**productos_categorias:** jerarquía categoría/subcategoría para enriquecer productos.
**clientes:** maestro de clientes con segmento y ubicación.
**territorios:** mapa de ClaveTerritorio → país y continente.
**campanas:** gasto de marketing por territorio/campaña.

**Contexto del negocio**

Tu director financiero busca responder dos preguntas centrales:

1. ¿Cuánto estamos ganando por país?
2. ¿Qué tan rentable es cada mercado considerando los gastos de marketing?

**Proceso a grandes pasos**

1. Explorar el esquema: Diagrama de Entidades y definición de esquema de Tablas (30–60 min).
2. Extraer y limpiar datos con consultas SQL y vistas (30–60 min).
3. Calcular KPIs financieros y guardarlos en vistas (60-90 min).
4. Validar resultados y QA (15–30 min).
5. Preparar outputs y resumen ejecutivo en formato CFI en Google Drive (30–60 min).

**💡 Recuerda:** los buenos analistas no lo saben todo, pero sí saben cómo encontrar respuestas.

# Parte 1: Explorar el esquema

## **Paso 1:** Imprime los 10 primeros renglones de cada tabla (`ventas_2017`, `productos`, `productos_categorias`, `territorios`, `campanas`).

### 1. Imprime los 10 primeros renglones de la tabla ventas_2017. 

In [ ]:
SELECT *
FROM productos
LIMIT 10

### 2. Imprime los 10 primeros renglones de la tabla productos. 

In [ ]:
SELECT *
FROM productos
LIMIT 10

### 3. Imprime los 10 primeros renglones de la tabla productos_categorias. 

In [ ]:
SELECT *
FROM productos_categorias
LIMIT 10

### 4. Imprime los 10 primeros renglones de la tabla territorios. 

In [ ]:
SELECT *
FROM territorios
LIMIT 10

### 5. Imprime los 10 primeros renglones de la tabla campanas. 

In [ ]:
SELECT *
FROM campanas
LIMIT 10

# Parte 2: Extraer y limpiar datos

**Paso 1: Extraer y limpiar datos**

Antes de calcular ingresos y rentabilidad, necesitamos construir una tabla base que combine la información clave de ventas, productos y territorios.

Piensa en:

- **Unión de Tablas:** Piensa qué datos necesitas tener en una sola tabla para responder a las preguntas del director financiero (¿cuánto se vende, a qué precio, con qué costo, y en qué país?).
- **Selección de columnas:** Asegúrate de incluir columnas que identifiquen la orden, el producto, su categoría, la campaña de marketing y el territorio.
- **Tratamiento de Valores Nulos:** Como en muchos sistemas reales, habrá valores faltantes: deberás decidir cómo tratarlos (ej. reemplazar nulos por 0 en ingresos o costos).

**Paso 2: Añade 2 columnas calculadas a tu query**

Para responder preguntas de negocio necesitamos números que nos digan dinero que entra y dinero que sale.

👉🏼 Tu tarea en este paso es crear dos nuevas columnas calculadas:

- `ingreso_total` → precio de cada producto × cantidad pedida.
- `costo_total` → costo de cada producto × cantidad pedida.

## 1. Union de tablas

Piensa qué datos necesitas tener en una sola tabla para responder a las preguntas del director financiero (¿cuánto se vende, a qué precio, con qué costo, y en qué país?).

**Instrucciones Generales** 

1. Une la tablas
    - `ventas_2017` v con `productos p`, 
    - `productos p` con `productos_categorias pc`, 
    - `ventas_2017 v` con `territorios t`

2. Incluye las siguientes columnas en el SELECT: `v.numero_pedido`, `v.clave_producto`, `p.nombre_producto`, `pc.clave_categoria`, `p.precio_producto`, `v.cantidad_pedido`, `p.costo_producto`, `t.pais`, `t.continente`, `v.clave_territorio`

3. Reemplaza valores nulos en las columnas de `precio`, `cantidad` y `costo`.

In [ ]:
SELECT
    v.numero_pedido,
    v.clave_producto,
    p.nombre_producto,
    pc.clave_categoria,
    COALESCE(p.precio_producto, 0)  AS precio_producto,
    COALESCE(v.cantidad_pedido, 0)  AS cantidad_pedido,
    COALESCE(p.costo_producto, 0)   AS costo_producto,
    t.pais,
    t.continente,
    v.clave_territorio
FROM ventas_2017 AS v
JOIN productos AS p
  ON v.clave_producto = p.clave_producto
LEFT JOIN productos_categorias AS pc
  ON p.clave_subcategoria = pc.clave_subcategoria
LEFT JOIN territorios AS t
  ON v.clave_territorio = t.clave_territorio;

## 2. Calculo de ingresos y costos

Tu tarea en este paso es crear dos nuevas columnas calculadas:

**Instrucciones Generales**

1. Añade la columna `ingreso_total` → precio de cada producto × cantidad pedida.
2. Añade la columna `costo_total` → costo de cada producto × cantidad pedida.
3. No se te olvide reemplazar nulos por ceros para evitar errores.

In [ ]:
SELECT
    v.numero_pedido,
    v.clave_producto,
    p.nombre_producto,
    pc.clave_categoria,
    COALESCE(p.precio_producto, 0) AS precio_producto,
    COALESCE(v.cantidad_pedido, 0) AS cantidad_pedido,
    COALESCE(p.costo_producto, 0)  AS costo_producto,
    t.pais,
    t.continente,
    v.clave_territorio,
    -- Cálculos
    COALESCE(p.precio_producto, 0) * COALESCE(v.cantidad_pedido, 0) AS ingreso_total,
    COALESCE(p.costo_producto, 0)  * COALESCE(v.cantidad_pedido, 0) AS costo_total
FROM ventas_2017 AS v
JOIN productos AS p
  ON v.clave_producto = p.clave_producto
LEFT JOIN productos_categorias AS pc
  ON p.clave_subcategoria = pc.clave_subcategoria
LEFT JOIN territorios AS t
  ON v.clave_territorio = t.clave_territorio;

# Parte 3: Calcular KPIs financieros